In [ ]:
# Best RNN hyperparameters embedded from Playground_Kai/checkpoints/best_hyperparams_rnn.yaml
BEST_HYPERPARAMS_YAML = """
lr: 0.0004584197485106983
weight_decay: 0.05407136503563504
hidden_size: 512
num_layers: 1
dropout: 0.44212708840605874
trial_val_cer: 91.4931
search_mode: two-phase
search_phase_found: C
num_coarse_trials: 30
num_fine_trials: 10
coarse_epochs: 10
fine_epochs: 10
num_trial_sessions: 8
fine_shrink: 3.0
tuned_at: '2026-03-07 15:38:16'
"""

# Train/val/test split embedded from config/user/single_user.yaml
SPLIT_YAML = """
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-03-1622764398-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626917264-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622889105-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-03-1622766673-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622861066-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627001995-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622884635-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626915176-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  val:
  - user: 89335547
    session: 2021-06-04-1622862148-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  test:
  - user: 89335547
    session: 2021-06-02-1622682789-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
"""

# RNN Raw Log-Spectrogram Notebook (Self-Contained)

This notebook reproduces the pipeline intent of:
`python -m Playground_Kai.train --model rnn --from-hyperparams ... --epochs 150`

The code is self-contained and does not import `Playground_Kai` or `emg2qwerty` modules at runtime.

Expected external inputs:
- Raw session HDF5 files under `data/`
- Installed packages from `Playground_Kai/requirements.txt`

In [ ]:
from __future__ import annotations

import csv
import json
import math
import random
import string
import time
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Sequence

import h5py
import Levenshtein
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
import yaml
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset


def seed_everything(seed: int = 13) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class CharacterSet:
    def __init__(self) -> None:
        chars = list(string.ascii_letters + string.digits + string.punctuation)
        special = ["Key.backspace", "Key.enter", "Key.space", "Key.shift"]
        self.allowed_keys = chars + special
        self.key_to_label = {k: i for i, k in enumerate(self.allowed_keys)}
        self.label_to_key = {i: k for k, i in self.key_to_label.items()}
        self.null_class = len(self.allowed_keys)
        self.num_classes = len(self.allowed_keys) + 1

    def keys_to_text(self, keys: Sequence[str]) -> str:
        out: list[str] = []
        for k in keys:
            if k == "Key.backspace":
                out.append("\b")
            elif k == "Key.enter":
                out.append("\n")
            elif k == "Key.space":
                out.append(" ")
            elif k == "Key.shift":
                out.append("\x11")
            elif len(k) == 1:
                out.append(k)
        return "".join(out)

    def labels_to_text(self, labels: Sequence[int]) -> str:
        keys = [self.label_to_key[int(x)] for x in labels if int(x) in self.label_to_key]
        return self.keys_to_text(keys)


CHARSET = CharacterSet()


@dataclass
class LabelData:
    text: str

    @classmethod
    def from_labels(cls, labels: Sequence[int]) -> "LabelData":
        return cls(text=CHARSET.labels_to_text(labels))

    @classmethod
    def from_keystrokes(cls, keystrokes: Sequence[dict[str, Any]], start_t: float, end_t: float) -> "LabelData":
        keys: list[str] = []
        for item in keystrokes:
            if item["start"] > end_t:
                break
            if item["start"] >= start_t:
                key = item["key"]
                if key in CHARSET.key_to_label:
                    keys.append(key)
        return cls(text=CHARSET.keys_to_text(keys))

    @property
    def labels(self) -> np.ndarray:
        labels: list[int] = []
        key_stream: list[str] = []
        for c in self.text:
            if c == "\b":
                key_stream.append("Key.backspace")
            elif c == "\n":
                key_stream.append("Key.enter")
            elif c == " ":
                key_stream.append("Key.space")
            elif c == "\x11":
                key_stream.append("Key.shift")
            else:
                key_stream.append(c)
        for key in key_stream:
            if key in CHARSET.key_to_label:
                labels.append(CHARSET.key_to_label[key])
        return np.asarray(labels, dtype=np.int32)

    def __len__(self) -> int:
        return len(self.text)


Transform = Callable[[Any], Any]


@dataclass
class ToTensor:
    fields: Sequence[str] = ("emg_left", "emg_right")
    stack_dim: int = 1

    def __call__(self, data: np.ndarray) -> torch.Tensor:
        return torch.stack([torch.as_tensor(data[f]) for f in self.fields], dim=self.stack_dim)


@dataclass
class ForEach:
    transform: Callable[[torch.Tensor], torch.Tensor]
    batch_dim: int = 1

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        return torch.stack([self.transform(t) for t in x.unbind(self.batch_dim)], dim=self.batch_dim)


@dataclass
class Compose:
    transforms: Sequence[Transform]

    def __call__(self, x: Any) -> Any:
        for t in self.transforms:
            x = t(x)
        return x


@dataclass
class RandomBandRotation:
    offsets: Sequence[int] = (-1, 0, 1)
    channel_dim: int = -1

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        offset = int(np.random.choice(self.offsets)) if len(self.offsets) > 0 else 0
        return x.roll(offset, dims=self.channel_dim)


@dataclass
class TemporalAlignmentJitter:
    max_offset: int
    stack_dim: int = 1

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        left, right = x.unbind(self.stack_dim)
        offset = int(np.random.randint(-self.max_offset, self.max_offset + 1))
        if offset > 0:
            left = left[offset:]
            right = right[:-offset]
        elif offset < 0:
            left = left[:offset]
            right = right[-offset:]
        return torch.stack([left, right], dim=self.stack_dim)


@dataclass
class LogSpectrogram:
    n_fft: int = 64
    hop_length: int = 16

    def __post_init__(self) -> None:
        self.spec = torchaudio.transforms.Spectrogram(n_fft=self.n_fft, hop_length=self.hop_length, normalized=True, center=False)

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        y = x.movedim(0, -1)
        y = self.spec(y)
        return torch.log10(y + 1e-6).movedim(-1, 0)


@dataclass
class SpecAugment:
    n_time_masks: int = 3
    time_mask_param: int = 25
    n_freq_masks: int = 2
    freq_mask_param: int = 4

    def __post_init__(self) -> None:
        self.time_mask = torchaudio.transforms.TimeMasking(self.time_mask_param, iid_masks=True)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(self.freq_mask_param, iid_masks=True)

    def __call__(self, specgram: torch.Tensor) -> torch.Tensor:
        x = specgram.movedim(0, -1)
        for _ in range(int(np.random.randint(self.n_time_masks + 1))):
            x = self.time_mask(x, mask_value=0.0)
        for _ in range(int(np.random.randint(self.n_freq_masks + 1))):
            x = self.freq_mask(x, mask_value=0.0)
        return x.movedim(-1, 0)


def build_train_transform() -> Compose:
    return Compose([
        ToTensor(fields=["emg_left", "emg_right"]),
        ForEach(RandomBandRotation(offsets=[-1, 0, 1])),
        TemporalAlignmentJitter(max_offset=120),
        LogSpectrogram(n_fft=64, hop_length=16),
        SpecAugment(n_time_masks=3, time_mask_param=25, n_freq_masks=2, freq_mask_param=4),
    ])


def build_eval_transform() -> Compose:
    return Compose([ToTensor(fields=["emg_left", "emg_right"]), LogSpectrogram(n_fft=64, hop_length=16)])


class EMGSessionData:
    GROUP = "emg2qwerty"

    def __init__(self, hdf5_path: Path) -> None:
        self.hdf5_path = Path(hdf5_path)
        self._file = h5py.File(self.hdf5_path, "r")
        grp = self._file[self.GROUP]
        self.timeseries = grp["timeseries"]
        self.metadata: dict[str, Any] = {}
        for key, val in grp.attrs.items():
            self.metadata[key] = json.loads(val) if key in {"keystrokes", "prompts"} else val

    def __enter__(self) -> "EMGSessionData":
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> None:
        self._file.close()

    def __len__(self) -> int:
        return len(self.timeseries)

    def __getitem__(self, key: slice | str) -> np.ndarray:
        return self.timeseries[key]

    def ground_truth(self, start_t: float, end_t: float) -> LabelData:
        return LabelData.from_keystrokes(self.metadata["keystrokes"], start_t, end_t)


@dataclass
class WindowedEMGDataset(Dataset):
    hdf5_path: Path
    window_length: int | None = None
    stride: int | None = None
    padding: tuple[int, int] = (1800, 200)
    jitter: bool = False
    transform: Transform = field(default_factory=ToTensor)

    def __post_init__(self) -> None:
        with EMGSessionData(self.hdf5_path) as session:
            self.session_length = len(session)
        self.window_length = self.window_length if self.window_length is not None else self.session_length
        self.stride = self.stride if self.stride is not None else self.window_length
        self.left_padding, self.right_padding = self.padding

    def __len__(self) -> int:
        return int(max(self.session_length - self.window_length, 0) // self.stride + 1)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        if not hasattr(self, "session"):
            self.session = EMGSessionData(self.hdf5_path)

        offset = idx * self.stride
        leftover = len(self.session) - (offset + self.window_length)
        if leftover < 0:
            raise IndexError(f"Index {idx} out of bounds")
        if leftover > 0 and self.jitter:
            offset += int(np.random.randint(0, min(self.stride, leftover)))

        window_start = max(offset - self.left_padding, 0)
        window_end = offset + self.window_length + self.right_padding
        window = self.session[window_start:window_end]

        emg = self.transform(window)
        timestamps = window["time"]
        start_t = float(timestamps[offset - window_start])
        end_t = float(timestamps[(offset + self.window_length - 1) - window_start])
        labels = torch.as_tensor(self.session.ground_truth(start_t, end_t).labels)
        return emg, labels

    @staticmethod
    def collate(samples: Sequence[tuple[torch.Tensor, torch.Tensor]]) -> dict[str, torch.Tensor]:
        inputs = [s[0] for s in samples]
        targets = [s[1] for s in samples]
        input_batch = nn.utils.rnn.pad_sequence(inputs)
        target_batch = nn.utils.rnn.pad_sequence(targets)
        input_lengths = torch.as_tensor([len(x) for x in inputs], dtype=torch.int32)
        target_lengths = torch.as_tensor([len(y) for y in targets], dtype=torch.int32)
        return {
            "inputs": input_batch,
            "targets": target_batch,
            "input_lengths": input_lengths,
            "target_lengths": target_lengths,
        }


def split_paths(data_root: Path, split_yaml: str) -> dict[str, list[Path]]:
    cfg = yaml.safe_load(split_yaml)
    out: dict[str, list[Path]] = {}
    for split in ("train", "val", "test"):
        paths: list[Path] = []
        for entry in cfg["dataset"].get(split, []):
            p = data_root / f"{entry['session']}.hdf5"
            if not p.exists():
                raise FileNotFoundError(f"Missing HDF5: {p}")
            paths.append(p)
        out[split] = paths
    return out


def get_dataloaders(data_root: Path, split_yaml: str, window_length: int, batch_size: int, num_workers: int) -> dict[str, DataLoader]:
    paths = split_paths(data_root, split_yaml)
    train_transform = build_train_transform()
    eval_transform = build_eval_transform()

    train_dataset = ConcatDataset([
        WindowedEMGDataset(p, window_length=window_length, stride=None, padding=(1800, 200), jitter=True, transform=train_transform)
        for p in paths["train"]
    ])
    val_dataset = ConcatDataset([
        WindowedEMGDataset(p, window_length=window_length, stride=None, padding=(1800, 200), jitter=False, transform=eval_transform)
        for p in paths["val"]
    ])
    test_dataset = ConcatDataset([
        WindowedEMGDataset(p, window_length=window_length, stride=None, padding=(1800, 200), jitter=False, transform=eval_transform)
        for p in paths["test"]
    ])

    persistent = num_workers > 0
    return {
        "train": DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=WindowedEMGDataset.collate, pin_memory=True, persistent_workers=persistent),
        "val": DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=WindowedEMGDataset.collate, pin_memory=True, persistent_workers=persistent),
        "test": DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=num_workers, collate_fn=WindowedEMGDataset.collate, pin_memory=True, persistent_workers=persistent),
    }


class SpectrogramNorm(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.channels = channels
        self.batch_norm = nn.BatchNorm2d(channels)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        T, N, bands, C, freq = inputs.shape
        assert self.channels == bands * C
        x = inputs.movedim(0, -1).reshape(N, bands * C, freq, T)
        x = self.batch_norm(x)
        return x.reshape(N, bands, C, freq, T).movedim(-1, 0)


class RotationInvariantMLP(nn.Module):
    def __init__(self, in_features: int, mlp_features: Sequence[int], pooling: str = "mean", offsets: Sequence[int] = (-1, 0, 1)) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        for out_features in mlp_features:
            layers.extend([nn.Linear(in_features, out_features), nn.ReLU()])
            in_features = out_features
        self.mlp = nn.Sequential(*layers)
        self.pooling = pooling
        self.offsets = offsets if len(offsets) > 0 else (0,)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.stack([x.roll(offset, dims=2) for offset in self.offsets], dim=2)
        x = self.mlp(x.flatten(start_dim=3))
        return x.max(dim=2).values if self.pooling == "max" else x.mean(dim=2)


class MultiBandRotationInvariantMLP(nn.Module):
    def __init__(self, in_features: int, mlp_features: Sequence[int], pooling: str = "mean", offsets: Sequence[int] = (-1, 0, 1), num_bands: int = 2, stack_dim: int = 2) -> None:
        super().__init__()
        self.num_bands = num_bands
        self.stack_dim = stack_dim
        self.mlps = nn.ModuleList([RotationInvariantMLP(in_features, mlp_features, pooling, offsets) for _ in range(num_bands)])

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        assert inputs.shape[self.stack_dim] == self.num_bands
        bands = inputs.unbind(self.stack_dim)
        outs = [mlp(x) for mlp, x in zip(self.mlps, bands)]
        return torch.stack(outs, dim=self.stack_dim)


class RNNEncoder(nn.Module):
    NUM_BANDS = 2

    def __init__(self, in_features: int = 528, mlp_features: Sequence[int] = (384,), hidden_size: int = 512, num_layers: int = 1, dropout: float = 0.2, electrode_channels: int = 16) -> None:
        super().__init__()
        self.spec_norm = SpectrogramNorm(channels=self.NUM_BANDS * electrode_channels)
        self.mlp = MultiBandRotationInvariantMLP(in_features=in_features, mlp_features=list(mlp_features), num_bands=self.NUM_BANDS)
        rnn_in = self.NUM_BANDS * list(mlp_features)[-1]
        self.rnn = nn.LSTM(input_size=rnn_in, hidden_size=hidden_size, num_layers=num_layers, bidirectional=True, dropout=dropout if num_layers > 1 else 0.0, batch_first=False)
        self.head = nn.Linear(2 * hidden_size, CHARSET.num_classes)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        x = self.spec_norm(inputs)
        x = self.mlp(x)
        x = x.flatten(start_dim=2).contiguous()
        x, _ = self.rnn(x)
        x = self.head(x)
        return F.log_softmax(x, dim=-1)


class CTCGreedyDecoder:
    def decode_batch(self, emissions: np.ndarray, emission_lengths: np.ndarray) -> list[LabelData]:
        out: list[LabelData] = []
        for i in range(emissions.shape[1]):
            e = emissions[: int(emission_lengths[i]), i]
            dec: list[int] = []
            prev = CHARSET.null_class
            for lab in e.argmax(-1):
                lab_i = int(lab)
                if lab_i != CHARSET.null_class and lab_i != prev:
                    dec.append(lab_i)
                prev = lab_i
            out.append(LabelData.from_labels(dec))
        return out


class CharacterErrorRates:
    def __init__(self) -> None:
        self.insertions = 0
        self.deletions = 0
        self.substitutions = 0
        self.target_len = 0

    def update(self, prediction: LabelData, target: LabelData) -> None:
        edits = Counter(op for op, _, _ in Levenshtein.editops(prediction.text, target.text))
        self.insertions += int(edits["insert"])
        self.deletions += int(edits["delete"])
        self.substitutions += int(edits["replace"])
        self.target_len += len(target)

    def compute(self) -> dict[str, float]:
        denom = max(self.target_len, 1)
        return {
            "CER": (self.insertions + self.deletions + self.substitutions) / denom * 100.0,
            "IER": self.insertions / denom * 100.0,
            "DER": self.deletions / denom * 100.0,
            "SER": self.substitutions / denom * 100.0,
        }


RESULTS_DIR = Path("results")


def make_run_id(model: str, num_channels: int, sampling_rate_hz: int, train_fraction: float) -> str:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    pct = int(round(train_fraction * 100))
    return f"{model.upper()}_{num_channels}ch_{sampling_rate_hz}hz_{pct}pct_{ts}"


def append_csv(path: Path, cols: list[str], row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=cols)
        if write_header:
            writer.writeheader()
        writer.writerow({c: row.get(c, "") for c in cols})


def log_epoch(run_id: str, model: str, epoch: int, train_loss: float, val_loss: float, val_cer: float) -> None:
    append_csv(RESULTS_DIR / f"results_curves_{model.upper()}.csv", ["run_id", "epoch", "train_loss", "val_loss", "val_cer"], {"run_id": run_id, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_cer": val_cer})


def log_summary(run_id: str, model: str, epochs: int, final_train_loss: float, final_val_loss: float, final_val_cer: float, test_cer: float, training_time_sec: float, notes: str = "") -> None:
    append_csv(RESULTS_DIR / f"results_summary_{model.upper()}.csv", ["run_id", "timestamp", "model", "epochs", "final_train_loss", "final_val_loss", "final_val_cer", "test_cer", "training_time_sec", "notes"], {"run_id": run_id, "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "model": model.upper(), "epochs": epochs, "final_train_loss": final_train_loss, "final_val_loss": final_val_loss, "final_val_cer": final_val_cer, "test_cer": test_cer, "training_time_sec": training_time_sec, "notes": notes})


@dataclass
class TrainConfig:
    epochs: int = 150
    batch_size: int = 32
    num_workers: int = 0
    window_length: int = 8000
    lr: float = 4.584197485106983e-4
    weight_decay: float = 5.407136503563504e-2
    min_lr: float = 1e-5
    warmup_epochs: int = 5
    hidden_size: int = 512
    num_layers: int = 1
    dropout: float = 0.44212708840605874
    data_root: Path = Path("data")
    checkpoint_dir: Path = Path("Playground_Kai/checkpoints")


def _lr_lambda(step: int, warmup_steps: int, total_steps: int, min_lr_ratio: float) -> float:
    if step < warmup_steps:
        return float(step) / max(1, warmup_steps)
    progress = float(step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr_ratio + (1.0 - min_lr_ratio) * cosine


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, criterion: nn.CTCLoss, device: torch.device, scheduler: torch.optim.lr_scheduler.LambdaLR) -> float:
    model.train()
    total_loss = 0.0
    for batch in loader:
        inputs = batch["inputs"].to(device)
        targets = batch["targets"].to(device)
        input_lengths = batch["input_lengths"].to(device)
        target_lengths = batch["target_lengths"].to(device)

        optimizer.zero_grad()
        emissions = model(inputs)
        loss = criterion(emissions, targets.transpose(0, 1), input_lengths, target_lengths)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        scheduler.step()
        total_loss += float(loss.item())

    return total_loss / max(1, len(loader))


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, decoder: CTCGreedyDecoder) -> tuple[float, dict[str, float]]:
    model.eval()
    criterion = nn.CTCLoss(blank=CHARSET.null_class, zero_infinity=True)
    cer_metric = CharacterErrorRates()
    total_loss = 0.0

    for batch in loader:
        inputs = batch["inputs"].to(device)
        targets = batch["targets"]
        input_lengths = batch["input_lengths"]
        target_lengths = batch["target_lengths"]

        with torch.backends.cudnn.flags(enabled=False):
            emissions = model(inputs)

        loss = criterion(emissions, targets.transpose(0, 1).to(device), input_lengths.to(device), target_lengths.to(device))
        total_loss += float(loss.item())

        preds = decoder.decode_batch(emissions.cpu().numpy(), input_lengths.numpy())
        targets_np = targets.numpy()
        target_lens = target_lengths.numpy()
        for i, pred in enumerate(preds):
            tgt = LabelData.from_labels(targets_np[: int(target_lens[i]), i])
            cer_metric.update(prediction=pred, target=tgt)

    return total_loss / max(1, len(loader)), cer_metric.compute()


def resolve_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data").exists():
        return cwd
    for parent in cwd.parents:
        if (parent / "data").exists():
            return parent
    return cwd


def run_training(cfg: TrainConfig) -> None:
    seed_everything(13)
    root = resolve_root()
    data_root = (root / cfg.data_root).resolve() if not cfg.data_root.is_absolute() else cfg.data_root
    checkpoint_dir = (root / cfg.checkpoint_dir).resolve() if not cfg.checkpoint_dir.is_absolute() else cfg.checkpoint_dir
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    print(f"Data root: {data_root}")
    print(f"Checkpoint dir: {checkpoint_dir}")

    loaders = get_dataloaders(data_root=data_root, split_yaml=SPLIT_YAML, window_length=cfg.window_length, batch_size=cfg.batch_size, num_workers=cfg.num_workers)
    print(f"Batches train/val/test: {len(loaders['train'])}/{len(loaders['val'])}/{len(loaders['test'])}")

    model = RNNEncoder(in_features=528, mlp_features=(384,), hidden_size=cfg.hidden_size, num_layers=cfg.num_layers, dropout=cfg.dropout, electrode_channels=16).to(device)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    criterion = nn.CTCLoss(blank=CHARSET.null_class, zero_infinity=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    steps_per_epoch = len(loaders["train"])
    total_steps = cfg.epochs * steps_per_epoch
    warmup_steps = cfg.warmup_epochs * steps_per_epoch
    min_lr_ratio = cfg.min_lr / cfg.lr
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda s: _lr_lambda(s, warmup_steps, total_steps, min_lr_ratio))

    run_id = make_run_id("RNN", num_channels=32, sampling_rate_hz=2000, train_fraction=0.89)
    decoder = CTCGreedyDecoder()
    best_cer = float("inf")
    best_path = checkpoint_dir / "best_rnn_528.pt"

    t0 = time.perf_counter()
    final_train_loss = float("nan")
    final_val_loss = float("nan")
    final_val_cer = float("nan")

    for epoch in range(cfg.epochs):
        e0 = time.perf_counter()
        train_loss = train_one_epoch(model, loaders["train"], optimizer, criterion, device, scheduler)
        val_loss, val_metrics = evaluate(model, loaders["val"], device, decoder)
        val_cer = float(val_metrics["CER"])

        final_train_loss = train_loss
        final_val_loss = val_loss
        final_val_cer = val_cer

        log_epoch(run_id, "RNN", epoch + 1, train_loss, val_loss, val_cer)

        if val_cer < best_cer:
            best_cer = val_cer
            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_cer": best_cer,
                "config": cfg.__dict__,
            }, best_path)
            tag = " [saved best]"
        else:
            tag = ""

        dt = time.perf_counter() - e0
        print(f"Epoch {epoch + 1:3d}/{cfg.epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_CER={val_cer:.2f}% | {dt:.1f}s{tag}")

    if best_path.exists():
        state = torch.load(best_path, map_location=device)
        model.load_state_dict(state["model"])
        print(f"Loaded best checkpoint from epoch {state['epoch'] + 1}")

    _, test_metrics = evaluate(model, loaders["test"], device, decoder)
    total_time = time.perf_counter() - t0

    print("Test metrics:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v:.2f}%")

    log_summary(run_id=run_id, model="RNN", epochs=cfg.epochs, final_train_loss=final_train_loss, final_val_loss=final_val_loss, final_val_cer=final_val_cer, test_cer=float(test_metrics.get("CER", float("nan"))), training_time_sec=total_time, notes="raw_logspectrogram_rnn")

    print(f"Training complete in {time.strftime('%H:%M:%S', time.gmtime(total_time))}")
    print(f"Best checkpoint: {best_path}")

In [ ]:
# Build runtime config from embedded best hyperparameters
best_hp = yaml.safe_load(BEST_HYPERPARAMS_YAML)
cfg = TrainConfig(
    epochs=150,
    batch_size=32,
    num_workers=0,
    window_length=8000,
    lr=float(best_hp["lr"]),
    weight_decay=float(best_hp["weight_decay"]),
    hidden_size=int(best_hp["hidden_size"]),
    num_layers=int(best_hp["num_layers"]),
    dropout=float(best_hp["dropout"]),
    data_root=Path("data"),
    checkpoint_dir=Path("Playground_Kai/checkpoints"),
)
cfg

In [ ]:
# Run this cell to train and evaluate
run_training(cfg)